# Microzone Cleanup Notebook

This notebook takes a microzones (MAZ) from WFRC + MAG and traffic analysis zone (TAZ) layer and fills in gaps, then merges them into neighbors without crossing over the taz boundry. Then checks for any maz that cross over taz then cuts and remerges them back into the proper maz.

Point workspace variable at the `.gdb` folder for your project

Ensure that microzone and traffic analysis zone are inside of your `.gdb` folder

Point `maz` variable at the name of microzone feature class 

Example: `maz = "microzone_layer"`

Point `taz` variable at the name of traffic analysis zone feature class

Example: `taz = "taz_layer"`




### Import Necessary Libraries

In [13]:
import arcpy

### Assign key Variables

In [14]:
workspace = r"D:\ABM-INP-Microzones\Scripts\Scratch\Microzones_Cleaner\Microzones_Cleaner.gdb"
maz = "microzones_Export"
taz = "TAZ_Export"
mag_maz ="Microzones_COMBINED"

### Point Arcpy at `.gdb` folder
Enable output overwrite

In [36]:
arcpy.env.workspace = workspace
arcpy.env.overwriteOutput = True

### Clean MAG Data
Copy the combined microzones layer, then keep only UTAH County in `_00_Microzones_Mag`. The `Microzones_COMBINED` source remains unchanged.

In [16]:
mag_maz_clean = "_00_Microzones_Mag"
mag_maz_layer = "mag_maz_clean_lyr"

arcpy.management.CopyFeatures(mag_maz, mag_maz_clean)
arcpy.management.MakeFeatureLayer(mag_maz_clean, mag_maz_layer)
arcpy.management.SelectLayerByAttribute(
    in_layer_or_view=mag_maz_layer,
    selection_type="NEW_SELECTION",
    where_clause="CO_NAME = 'UTAH'"
)

arcpy.management.SelectLayerByAttribute(
    in_layer_or_view=mag_maz_layer,
    selection_type="SWITCH_SELECTION"
)
deleted_count = int(arcpy.management.GetCount(mag_maz_layer)[0])
arcpy.management.DeleteRows(mag_maz_layer)

arcpy.management.Delete(mag_maz_layer)
print(f" deleted {deleted_count} non-UTAH rows.")

 deleted 2216 non-UTAH rows.


### Assign Microzone Ids for MAG

Rename the Microzone field to match WFRC convention so it auto joins

In [17]:
arcpy.management.AlterField(
    in_table=mag_maz_clean,
    field="maz_id",
    new_field_name="MAZID",
    new_field_alias = "MAZID"
)

code_block = """
rec = 0
def autoIncrement():
    global rec
    pStart = 11000 
    if (rec == 0): 
        rec = pStart 
    else: 
        rec += 1 
    return rec
"""

arcpy.management.CalculateField(
    in_table=mag_maz_clean,
    field="MAZID",
    expression="autoIncrement()",
    expression_type="PYTHON3",
    code_block=code_block
)

<Result 'D:\\ABM-INP-Microzones\\Scripts\\Scratch\\Microzones_Cleaner\\Microzones_Cleaner.gdb\\_00_Microzones_Mag'>

### Join MAG Microzones to WFRC Microzones

In [18]:
maz_comb = "_01_Microzones_combined"

arcpy.management.Merge([maz,mag_maz_clean],maz_comb)

    

<Result 'D:\\ABM-INP-Microzones\\Scripts\\Scratch\\Microzones_Cleaner\\Microzones_Cleaner.gdb\\_01_Microzones_combined'>

### Update Plan Area Field MAG Zones

In [19]:
maz_comb_lyr = "maz_comb_lyr"

arcpy.management.MakeFeatureLayer(maz_comb, maz_comb_lyr)

# 2. Select only the features where CO_NAME is UTAH
arcpy.management.SelectLayerByAttribute(
    in_layer_or_view=maz_comb_lyr,
    selection_type="NEW_SELECTION",
    where_clause="CO_NAME = 'UTAH'"
)

# 3. Update PLANAREA to 'MAG' for the selected features
arcpy.management.CalculateField(
    in_table=maz_comb_lyr,
    field="PLANAREA",
    expression="'MAG'",
    expression_type="PYTHON3"
)

<Result 'maz_comb_lyr'>

### Run Identity Tool

This will fill in all gaps between polygons

Creates a new feature class called `maz_i` which has more features than orignial maz becuase these are all new filled in featuers

Original Microzones total = 14,380

MAZID"0" = 863


New Zones = 878

MAZ identified  total = 15,258

In [20]:
maz_i = "_02_maz_i"

arcpy.analysis.Identity(
    in_features=taz, 
    identity_features=maz_comb, 
    out_feature_class=maz_i)

<Result 'D:\\ABM-INP-Microzones\\Scripts\\Scratch\\Microzones_Cleaner\\Microzones_Cleaner.gdb\\_02_maz_i'>

### Multipart to Singlepart

Identity will treat some new features a single one and this function will break them back into their own

Creates a new feature class called `maz_filled` with all gaps filled in as unique features

MAZ Filled total = 16,194

New Zones = 1,236

MAZID"0" = 1,795

NEW MAZID"0" = 932

Missing duplicate Zones = (new zones [16,194]) - (og zones [14,380]) = 1,814 - (MAZID"0"[1,795]) = 19
The 19 duplicate zones will be filterd out later when I look for sliver zones


In [21]:
maz_filled = "_03_maz_filled"
arcpy.management.MultipartToSinglepart(
    in_features= maz_i,
    out_feature_class=maz_filled)

<Result 'D:\\ABM-INP-Microzones\\Scripts\\Scratch\\Microzones_Cleaner\\Microzones_Cleaner.gdb\\_03_maz_filled'>

### Create new field for filled gap polygon

All new polygons created from the identify and multipart functions will have `MAZID = 0` so to easily identify them later I will make a dummy column for it
Flags 1,793 polygons


In [22]:
arcpy.management.CalculateField(
    in_table=maz_filled,
    field="gap_flag",
    expression="1 if (!MAZID! == 0 or !MAZID! is None) else 0",
    expression_type="PYTHON3",
    field_type="SHORT"
)

<Result 'D:\\ABM-INP-Microzones\\Scripts\\Scratch\\Microzones_Cleaner\\Microzones_Cleaner.gdb\\_03_maz_filled'>

### Calculate Poslby Posper and Perm/Area Score

In [23]:
pp_code = """
import math
def calc_pp(area, perimeter, current_pp):
    # If it already has a value, keep it (optional), or overwrite if None
    if current_pp is not None:
        return current_pp
    if perimeter == 0 or perimeter is None:
        return 0
    return (4 * math.pi * area) / (perimeter ** 2)
"""

arcpy.management.CalculateField(
    in_table=maz_filled, # Update this variable if you run it later in the script (e.g., maz_sliver)
    field="PolsbyPopper",
    expression="calc_pp(!Shape_Area!, !Shape_Length!, !PolsbyPopper!)",
    expression_type="PYTHON3",
    code_block=pp_code
)

# 2. Calculate Perimeter to Area Ratio
pa_code = """
def calc_pa(area, perimeter, current_pa):
    if current_pa is not None:
        return current_pa
    if area == 0 or area is None:
        return 0
    return perimeter / area
"""

arcpy.management.CalculateField(
    in_table=maz_filled, 
    field="PerimtertoArea",
    expression="calc_pa(!Shape_Area!, !Shape_Length!, !PerimtertoArea!)",
    expression_type="PYTHON3",
    code_block=pa_code
)

<Result 'D:\\ABM-INP-Microzones\\Scripts\\Scratch\\Microzones_Cleaner\\Microzones_Cleaner.gdb\\_03_maz_filled'>

## Identify zones for review

In addition to gap zones we want to flag sliver zones

Shape Area is less than 10,000

AND 

PolsbyPopper is less than .03

OR 

PerimtertoArea is less than .03

OR

Shape Area < 1,000


**This will flag 1,995 zones as sliver zones to be merged into the neighboring microzones**

In [24]:
maz_sliver = "_04_maz_sliver"


arcpy.management.CopyFeatures(
    in_features= maz_filled,
    out_feature_class=maz_sliver
)

code_block = """
def check_sliver(pp, pa, area, gap):
    # Guard clause: large features (highways) are never slivers
    if area is not None and area >= 10000:
        return 0
    
    # Check sliver criteria for features under 10,000
    if pp is not None and pp < 0.03:
        return 1
    if pa is not None and pa < 0.03:
        return 1
    if area is not None and area < 1000:
        return 1
    if gap == 1:
        return 1
    return 0
"""

arcpy.management.CalculateField(
    in_table=maz_sliver,
    field="Sliver_Flag",
    expression="check_sliver(!PolsbyPopper!, !PerimtertoArea!, !Shape_Area!, !gap_flag!)",
    expression_type="PYTHON3",
    code_block=code_block,
    field_type="SHORT"
)


<Result 'D:\\ABM-INP-Microzones\\Scripts\\Scratch\\Microzones_Cleaner\\Microzones_Cleaner.gdb\\_04_maz_sliver'>

### Additional layer of just sliver zones for eye test

In [25]:
slivers = "_05_iso_slivers"

arcpy.management.MakeFeatureLayer(
    in_features=maz_sliver, 
    out_layer=slivers
)

arcpy.management.SelectLayerByAttribute(
    in_layer_or_view=slivers, 
    selection_type="NEW_SELECTION", 
    where_clause="Sliver_Flag = 1"
)

arcpy.management.CopyFeatures(
    in_features=slivers,
    out_feature_class=slivers
)

<Result 'D:\\ABM-INP-Microzones\\Scripts\\Scratch\\Microzones_Cleaner\\Microzones_Cleaner.gdb\\_05_iso_slivers'>

### Create Taz vector lines

Eliminate tool requires vector lines to use as a boarder so we dont merge between TAZ

In [26]:
taz_lines = "_06_taz_lines"
arcpy.management.PolygonToLine(taz, taz_lines)

<Result 'D:\\ABM-INP-Microzones\\Scripts\\Scratch\\Microzones_Cleaner\\Microzones_Cleaner.gdb\\_06_taz_lines'>

### Run Eliminate

Merges the flagged sliver and gap zones into neighboring microzones. `taz_lines` is passed as the exclusion feature so nothing merges across a TAZ boundary.

Some flagged zones may be left unmerged (for example a sliver whose only neighbors are across a TAZ line, or whose neighbors are also flagged). The next steps re-check and retry those.

Because `_03_maz_filled` came from an Identity with the TAZ as the *input* feature, every polygon is already cut exactly at the TAZ boundary. Eliminate only dissolves polygons together inside a TAZ, so **no geometry is moved and no new crossovers are introduced**.

In [27]:
maz_cleaned = "_07_maz_cleaned"
elim_layer = "temp_elim_layer"

arcpy.management.MakeFeatureLayer(
    in_features=maz_sliver, 
    out_layer=elim_layer
)

arcpy.management.SelectLayerByAttribute(elim_layer, "NEW_SELECTION", 'Sliver_Flag = 1')

arcpy.management.Eliminate(
    in_features=elim_layer, 
    out_feature_class=maz_cleaned,
    ex_features=taz_lines
)

<Result 'D:\\ABM-INP-Microzones\\Scripts\\Scratch\\Microzones_Cleaner\\Microzones_Cleaner.gdb\\_07_maz_cleaned'>

### Helper to (re)calculate sliver flags

Shape metrics are **recalculated from the current geometry every time** (the earlier `calc_pp` / `calc_pa` only filled nulls, so values went stale after Eliminate). The sliver rule is the same one used in `_04_maz_sliver`.

This is the only function in the notebook because it is used several times.

In [ ]:
metric_code = """
import math
def pp(area, perim):
    return 0 if not perim else (4 * math.pi * area) / (perim ** 2)
def pa(area, perim):
    return 0 if not area else perim / area
def check_sliver(pp, pa, area, gap):
    if area >= 10000:
        return 0
    if pp < 0.03 or pa < 0.03 or area < 1000 or gap == 1:
        return 1
    return 0
"""

def flag_slivers(fc):
    """Recalculate PolsbyPopper, PerimtertoArea and Sliver_Flag from current geometry. Returns sliver count."""
    arcpy.management.CalculateField(fc, "PolsbyPopper", "pp(!shape.area!, !shape.length!)", "PYTHON3", metric_code)
    arcpy.management.CalculateField(fc, "PerimtertoArea", "pa(!shape.area!, !shape.length!)", "PYTHON3", metric_code)
    arcpy.management.CalculateField(
        fc, "Sliver_Flag",
        "check_sliver(!PolsbyPopper!, !PerimtertoArea!, !shape.area!, !gap_flag!)",
        "PYTHON3", metric_code, field_type="SHORT"
    )
    lyr = "temp_sliver_count"
    arcpy.management.MakeFeatureLayer(fc, lyr, "Sliver_Flag = 1")
    n = int(arcpy.management.GetCount(lyr)[0])
    arcpy.management.Delete(lyr)
    return n

### Re-check slivers after the first Eliminate

Copy `_07_maz_cleaned`, recalculate the metrics from the current geometry and count how many sliver zones are still left.

In [ ]:
maz_recheck = "_08_maz_recheck"
arcpy.management.CopyFeatures(maz_cleaned, maz_recheck)
left = flag_slivers(maz_recheck)
print(f"Sliver zones left after first Eliminate: {left}")

### Retry Eliminate on any leftover slivers

Each pass merges what it can (still blocked by `taz_lines`), recalculates the flags, and tries again, because merging one sliver can give its neighbor a valid target on the next pass. Stops when no slivers are left, when a pass makes no progress, or after 5 passes.

Nothing is moved, so no new crossovers or new slivers are created. If slivers remain after this loop, the final checks below will fail loudly.

In [ ]:
current = maz_recheck
prev_left = None

for i in range(1, 6):
    layer = f"temp_pass_{i}"
    arcpy.management.MakeFeatureLayer(current, layer)
    arcpy.management.SelectLayerByAttribute(layer, "NEW_SELECTION", "Sliver_Flag = 1")
    n_sel = int(arcpy.management.GetCount(layer)[0])
    print(f"Pass {i}: {n_sel} sliver zones selected")

    if n_sel == 0 or n_sel == prev_left:
        arcpy.management.Delete(layer)
        break

    out = f"_09_maz_pass_{i}"
    arcpy.management.Eliminate(in_features=layer, out_feature_class=out, ex_features=taz_lines)
    arcpy.management.Delete(layer)
    prev_left = flag_slivers(out)
    print(f"        -> {prev_left} sliver zones left, {int(arcpy.management.GetCount(out)[0])} zones total")
    current = out

maz_final_cleaned = "_15_maz_final_cleaned"
arcpy.management.CopyFeatures(current, maz_final_cleaned)
print(f"Final layer: {maz_final_cleaned} ({int(arcpy.management.GetCount(maz_final_cleaned)[0])} zones)")

### Diagnostic: list the leftover sliver zones

Lists every sliver zone still in `_15_maz_final_cleaned`, smallest first, and shows **why** `Eliminate` could not merge each one. For each sliver it counts the neighbors that share a border with it (length > 0), split into:

- **same-TAZ, not a sliver**: a valid merge target. If this is > 0, Eliminate should have worked.
- **same-TAZ, also a sliver**: the sliver is stuck behind other slivers.
- **other TAZ**: neighbors it must *not* merge into.

`outside` means the zone overlaps no TAZ by more than 0.01 m² (the zones final check 2 reports as outside every TAZ). Also saves `_17_leftover_slivers` so the zones can be inspected on the map.

In [ ]:
if "maz_uid" not in [f.name for f in arcpy.ListFields(maz_final_cleaned)]:
    arcpy.management.AddField(maz_final_cleaned, "maz_uid", "LONG")
arcpy.management.CalculateField(maz_final_cleaned, "maz_uid", "!OBJECTID!", "PYTHON3")

leftovers = "_17_leftover_slivers"
arcpy.analysis.Select(maz_final_cleaned, leftovers, "Sliver_Flag = 1")

# Shared borders between every pair of touching zones
nbr_tbl = "_18_maz_neighbors"
arcpy.analysis.PolygonNeighbors(
    maz_final_cleaned, nbr_tbl, ["maz_uid", "TAZID", "Sliver_Flag"],
    "NO_AREA_OVERLAP", "BOTH_SIDES", "", "METERS", "SQUARE_METERS"
)
nbrs = {}
with arcpy.da.SearchCursor(nbr_tbl, ["src_maz_uid", "src_TAZID", "nbr_TAZID", "nbr_Sliver_Flag", "LENGTH"]) as rows:
    for s, s_taz, n_taz, n_sliver, length in rows:
        if length and length > 0:  # ignore corner-only (point) contacts
            nbrs.setdefault(s, []).append((s_taz, n_taz, n_sliver, length))

# Overlap with any TAZ (same test as final check 2)
xt = "_19_leftover_tabulate"
arcpy.analysis.TabulateIntersection(leftovers, "maz_uid", taz, xt, ["TAZID"])
taz_overlap = {}
with arcpy.da.SearchCursor(xt, ["maz_uid", "AREA"]) as rows:
    for uid, area in rows:
        taz_overlap[uid] = taz_overlap.get(uid, 0) + area

records = []
with arcpy.da.SearchCursor(leftovers, ["maz_uid", "MAZID", "TAZID", "gap_flag", "SHAPE@AREA"]) as rows:
    for uid, mazid, tazid, gap, area in rows:
        n = nbrs.get(uid, [])
        records.append({
            "uid": uid, "mazid": mazid, "taz": tazid, "gap": gap, "area": area,
            "outside": taz_overlap.get(uid, 0) <= 0.01,
            "ok_nbr": sum(1 for s, t, sl, l in n if t == s and sl == 0),
            "sliver_nbr": sum(1 for s, t, sl, l in n if t == s and sl == 1),
            "other_taz_nbr": sum(1 for s, t, sl, l in n if t != s),
        })
records.sort(key=lambda r: r["area"])

print(f"{'uid':>7} {'MAZID':>7} {'TAZID':>6} {'gap':>3} {'area_m2':>10} {'outside':>7} {'ok_nbr':>6} {'sl_nbr':>6} {'oth_taz':>7}")
for r in records:
    print(f"{r['uid']:>7} {r['mazid']:>7} {r['taz']:>6} {r['gap']:>3} {r['area']:>10.4f} {str(r['outside']):>7} "
          f"{r['ok_nbr']:>6} {r['sliver_nbr']:>6} {r['other_taz_nbr']:>7}")

total = len(records)
print(f"\nTotal leftover slivers: {total}")
print(f"  outside every TAZ (final check 2): {sum(r['outside'] for r in records)}")
print(f"  < 1 m2: {sum(r['area'] < 1 for r in records)}   < 10 m2: {sum(r['area'] < 10 for r in records)}   "
      f"< 50 m2: {sum(r['area'] < 50 for r in records)}   >= 50 m2: {sum(r['area'] >= 50 for r in records)}")
print(f"  have a same-TAZ non-sliver neighbor (Eliminate should have worked): {sum(r['ok_nbr'] > 0 for r in records)}")
print(f"  only same-TAZ neighbors are other slivers: {sum(r['ok_nbr'] == 0 and r['sliver_nbr'] > 0 for r in records)}")
print(f"  no same-TAZ neighbor at all:               {sum(r['ok_nbr'] == 0 and r['sliver_nbr'] == 0 for r in records)}")

### Final check 1: zero sliver zones

Recalculate the metrics on `_15_maz_final_cleaned` from scratch and assert none are flagged. Fails if any sliver or unfilled gap zone remains.

In [ ]:
final_slivers = flag_slivers(maz_final_cleaned)
print(f"Sliver zones in final layer: {final_slivers}")
assert final_slivers == 0, f"{final_slivers} sliver zones remain in {maz_final_cleaned} (inspect Sliver_Flag = 1)"

### Final check 2: zero zones crossing a TAZ boundary

`TabulateIntersection` measures how much of each microzone's area falls inside each TAZ. A microzone passes only if it overlaps **exactly one** TAZ (ignoring overlaps under 0.01 m² as edge noise), and that TAZ matches its `TAZID` attribute. Zones that overlap no TAZ at all also fail.

In [ ]:
if "maz_uid" not in [f.name for f in arcpy.ListFields(maz_final_cleaned)]:
    arcpy.management.AddField(maz_final_cleaned, "maz_uid", "LONG")
arcpy.management.CalculateField(maz_final_cleaned, "maz_uid", "!OBJECTID!", "PYTHON3")

xtab = "_16_maz_taz_tabulate"
arcpy.analysis.TabulateIntersection(maz_final_cleaned, "maz_uid", taz, xtab, ["TAZID"])

overlaps = {}
with arcpy.da.SearchCursor(xtab, ["maz_uid", "TAZID", "AREA"]) as rows:
    for uid, tazid, area in rows:
        if area > 0.01:
            overlaps.setdefault(uid, set()).add(tazid)

with arcpy.da.SearchCursor(maz_final_cleaned, ["maz_uid", "TAZID"]) as rows:
    attr_taz = dict(rows)

crossing = [u for u, t in overlaps.items() if len(t) > 1]
outside = [u for u in attr_taz if u not in overlaps]
mismatch = [u for u, t in overlaps.items() if len(t) == 1 and attr_taz[u] not in t]

print(f"Zones crossing a TAZ boundary: {len(crossing)}")
print(f"Zones outside every TAZ:       {len(outside)}")
print(f"Zones whose TAZID attribute does not match their TAZ: {len(mismatch)}")

assert len(crossing) == 0, f"{len(crossing)} zones cross a TAZ boundary (maz_uid: {crossing[:10]})"
assert len(outside) == 0, f"{len(outside)} zones fall outside every TAZ (maz_uid: {outside[:10]})"
assert len(mismatch) == 0, f"{len(mismatch)} zones have a TAZID attribute that differs from their TAZ (maz_uid: {mismatch[:10]})"
print("PASS: 0 sliver zones and 0 TAZ crossovers")

### Final Cleaned Layer

**_15_maz_final_cleaned**

Verified by the two assert cells above:

- No sliver or unfilled gap zones
- No microzones that cross over TAZ boundaries

### Summary stats for microzone counts per TAZ, county, and city

For each aggregation level, this summarizes the distribution of microzone counts across areas: number of areas, minimum, maximum, mean, and median microzones per area. Areas with a null grouping value are excluded and reported.

In [38]:
from collections import Counter
from statistics import mean, median

summary_fields = [
    ("TAZ", "TAZID"),
    ("County", "CO_NAME"),
    ("City", "CITY_NAME"),
]

for area_name, field_name in summary_fields:
    microzones_per_area = Counter()
    missing_area_count = 0

    with arcpy.da.SearchCursor(maz_final_cleaned, [field_name]) as rows:
        for (area_value,) in rows:
            if area_value is None:
                missing_area_count += 1
            else:
                microzones_per_area[area_value] += 1

    counts = list(microzones_per_area.values())
    if not counts:
        raise ValueError(f"No non-null {field_name} values found in {maz_final_cleaned}.")

    print(f"{area_name} ({field_name}) microzones per area:")
    print(f"  Area count: {len(counts)}")
    print(f"  Min: {min(counts)}")
    print(f"  Max: {max(counts)}")
    print(f"  Mean: {mean(counts):.2f}")
    print(f"  Median: {median(counts):.2f}")
    if missing_area_count:
        print(f"  Microzones skipped (null {field_name}): {missing_area_count}")

TAZ (TAZID) microzones per area:
  Area count: 3562
  Min: 1
  Max: 38
  Mean: 4.03
  Median: 4.00
County (CO_NAME) microzones per area:
  Area count: 5
  Min: 531
  Max: 5484
  Mean: 2872.40
  Median: 1942.00
City (CITY_NAME) microzones per area:
  Area count: 109
  Min: 1
  Max: 1464
  Mean: 131.76
  Median: 73.00
